In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("../data/cleaned_aqi_final.csv")

df['Date'] = pd.to_datetime(df['Date'])

# sort by date
df = df.sort_values('Date').reset_index(drop=True)

print("Original data shape:", df.shape)

# =========================
# ADVANCED FEATURES
# =========================
df['AQI_lag1'] = df['AQI'].shift(1)
df['AQI_lag3'] = df['AQI'].shift(3)
df['AQI_lag7'] = df['AQI'].shift(7)

df['PM25_roll7'] = df['PM2.5'].rolling(7).mean()
df['PM10_roll7'] = df['PM10'].rolling(7).mean()

df = df.dropna()

print("After feature engineering:", df.shape)

# =========================
# FEATURES
# =========================
features = [
    'PM2.5',
    'PM10',
    'NO',
    'NO2',
    'NOx',
    'NH3',
    'CO',
    'SO2',
    'O3',
    'Benzene',
    'Toluene',
    'AQI_lag1',
    'AQI_lag3',
    'AQI_lag7',
    'PM25_roll7',
    'PM10_roll7'
]

target = 'AQI'

# =========================
# SCALE FEATURES
# =========================
scaler = MinMaxScaler()

scaled_features = scaler.fit_transform(df[features])

# =========================
# CREATE 30-DAY SEQUENCES
# =========================
SEQ_LENGTH = 30

X_sequences = []
y_targets = []
target_dates = []

for i in range(SEQ_LENGTH, len(scaled_features)):
    X_sequences.append(
        scaled_features[i-SEQ_LENGTH:i]
    )

    y_targets.append(df[target].iloc[i])
    target_dates.append(df['Date'].iloc[i])

X_sequences = np.array(X_sequences)
y_targets = np.array(y_targets)

print("Sequence input shape:", X_sequences.shape)
print("Target shape:", y_targets.shape)

# =========================
# SAVE
# =========================
np.save("../data/X_sequences_30.npy", X_sequences)
np.save("../data/y_targets_30.npy", y_targets)

sequence_dates = pd.DataFrame({
    'Date': target_dates
})

sequence_dates.to_csv("../data/sequence_dates_30.csv", index=False)

print("Saved successfully.")
   

Original data shape: (1827, 15)
After feature engineering: (1820, 20)
Sequence input shape: (1790, 30, 16)
Target shape: (1790,)
Saved successfully.
